In [1]:
import os

print("Current directory:", os.getcwd())
print("Project folders:", os.listdir())

Current directory: /Users/masomashoayb/Desktop/pyspark-nyc-taxi-pipeline
Project folders: ['screenshots', '.DS_Store', 'flight_data_pipeline.ipynb', 'output', 'README.md', '.gitignore', '.ipynb_checkpoints', 'data']


In [2]:
import shutil

total, used, free = shutil.disk_usage(os.getcwd())

print(f"Free storage: {free / (1024**3):.2f} GB")

Free storage: 60.09 GB


In [3]:
import requests
import os

# Dataset URL
url = "https://data.mathvsvibes.com/bts/bts_2025.parquet"

# Save location
filepath = "data/bts_2025.parquet"

# Download the file
response = requests.get(url, stream=True, timeout=60)
response.raise_for_status()

with open(filepath, "wb") as file:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            file.write(chunk)

print("Download completed!")
print("File size:", round(os.path.getsize(filepath) / (1024**2), 2), "MB")

/Users/masomashoayb/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Download completed!
File size: 80.72 MB


In [4]:
import os

print("Files in data folder:", os.listdir("data"))

print(
    "Dataset size:",
    round(os.path.getsize("data/bts_2025.parquet") / (1024**2), 2),
    "MB"
)

Files in data folder: ['bts_2024.parquet', 'bts_2012.parquet', 'bts_2013.parquet', 'bts_2025.parquet', 'bts_2011.parquet', 'bts_2018.parquet', 'bts_2019.parquet', 'bts_2010.parquet', 'bts_2015.parquet', 'bts_2023.parquet', 'bts_2022.parquet', 'bts_2014.parquet', 'bts_2020.parquet', 'bts_2016.parquet', 'bts_2017.parquet', 'bts_2021.parquet']
Dataset size: 80.72 MB


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Flight_Data_Pipeline")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark session created successfully!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 19:54:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/24 19:54:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.0.5
Spark session created successfully!


In [6]:
flights = spark.read.parquet("data/bts_2025.parquet")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [7]:
flights.printSchema()
flights.show(5, truncate=False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- op_carrier: string (nullable = true)
 |-- tail_num: string (nullable = true)
 |-- flight_num: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- crs_dep_time: string (nullable = true)
 |-- crs_arr_time: string (nullable = true)
 |-- dep_time: string (nullable = true)
 |-- arr_time: string (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- cancelled: integer (nullable = true)
 |-- diverted: integer (nullable = true)
 |-- carrier_delay: double (nullable = true)
 |-- weather_delay: double (nullable = true)
 |-- nas_delay: double (nullable = true)
 |-- security_delay: double (nullable = true)
 |-- late_aircraft_delay: double (nullable = true)
 |-- crs_elapsed: double (nullable = true)
 |-- actual_elapsed: double (

In [8]:
# Count the total number of flight records
total_flights = flights.count()

print("Total flights:", f"{total_flights:,}")

Total flights: 7,001,619


In [9]:
# Display all column names
print("Number of columns:", len(flights.columns))

print("\nColumn names:")
print(flights.columns)

Number of columns: 24

Column names:
['year', 'month', 'day_of_month', 'day_of_week', 'op_carrier', 'tail_num', 'flight_num', 'origin', 'dest', 'crs_dep_time', 'crs_arr_time', 'dep_time', 'arr_time', 'dep_delay', 'arr_delay', 'cancelled', 'diverted', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay', 'crs_elapsed', 'actual_elapsed']


In [10]:
from pyspark.sql.functions import col, sum

missing_values = flights.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in flights.columns
])

missing_values.show(truncate=False)

[Stage 5:=======>                                                   (1 + 7) / 8]

+----+-----+------------+-----------+----------+--------+----------+------+----+------------+------------+--------+--------+---------+---------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----------+--------------+
|year|month|day_of_month|day_of_week|op_carrier|tail_num|flight_num|origin|dest|crs_dep_time|crs_arr_time|dep_time|arr_time|dep_delay|arr_delay|cancelled|diverted|carrier_delay|weather_delay|nas_delay|security_delay|late_aircraft_delay|crs_elapsed|actual_elapsed|
+----+-----+------------+-----------+----------+--------+----------+------+----+------------+------------+--------+--------+---------+---------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----------+--------------+
|0   |0    |0           |0          |0         |12501   |0         |0     |0   |0           |0           |98170   |104608  |98591    |122135   |0        |0       |0            |0            |0        |0      

In [11]:
import requests

url = "https://data.mathvsvibes.com/bts/"

response = requests.get(url, timeout=30)

print("Website status:", response.status_code)
print(response.text[:2000])

Website status: 404
<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <link rel="icon" href="https://www.cloudflare.com/favicon.ico" />
    <title>Not Found</title>
    <style>
      body {
        font-family: system-ui;
        font-weight: 300;
        font-size: 1.25rem;
        color: #36393a;
        display: flex;
        align-items: center;
        justify-content: center;
      }
      main {
        max-width: 1200px;
        margin-top: 120px;
        display: flex;
        flex-wrap: wrap;
        align-items: center;
        justify-content: center;
      }
      #text {
        max-width: 60%;
        margin-left: 1rem;
        margin-right: 1rem;
      }
      main > section > div {
        margin-bottom: 3.25rem;
      }
      svg {
        margin-left: 2rem;
      }
      @keyframes eye-1 {
        0% {
          transform: translateX(0);
        }
        10%,
      

In [12]:
import requests

for year in range(2010, 2026):
    url = f"https://data.mathvsvibes.com/bts/bts_{year}.parquet"

    response = requests.head(url, allow_redirects=True, timeout=20)

    size_gb = int(response.headers.get("content-length", 0)) / (1024**3)

    print(year, response.status_code, f"{size_gb:.3f} GB")

2010 200 0.065 GB
2011 200 0.063 GB
2012 200 0.065 GB
2013 200 0.070 GB
2014 200 0.064 GB
2015 200 0.064 GB
2016 200 0.062 GB
2017 200 0.063 GB
2018 200 0.089 GB
2019 200 0.083 GB
2020 200 0.049 GB
2021 200 0.068 GB
2022 200 0.074 GB
2023 200 0.077 GB
2024 200 0.080 GB
2025 200 0.079 GB


In [13]:
import requests
from pathlib import Path

data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

for year in range(2010, 2025):
    file_path = data_folder / f"bts_{year}.parquet"

    if file_path.exists():
        print(f"{year}: Already downloaded, skipping.")
        continue

    url = f"https://data.mathvsvibes.com/bts/bts_{year}.parquet"

    print(f"Downloading {year}...")

    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with open(str(file_path) + ".part", "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

    Path(str(file_path) + ".part").rename(file_path)
    print(f"{year}: Download complete.")

print("All downloads finished.")

2010: Already downloaded, skipping.
2011: Already downloaded, skipping.
2012: Already downloaded, skipping.
2013: Already downloaded, skipping.
2014: Already downloaded, skipping.
2015: Already downloaded, skipping.
2016: Already downloaded, skipping.
2017: Already downloaded, skipping.
2018: Already downloaded, skipping.
2019: Already downloaded, skipping.
2020: Already downloaded, skipping.
2021: Already downloaded, skipping.
2022: Already downloaded, skipping.
2023: Already downloaded, skipping.
2024: Already downloaded, skipping.
All downloads finished.


In [14]:
from pyspark.sql.functions import *


In [15]:
import builtins

total_size = builtins.sum(
    file.stat().st_size
    for file in data_folder.glob("*.parquet")
)

print(f"Total dataset size: {total_size / (1024**3):.3f} GiB")
print(f"Number of yearly files: {len(list(data_folder.glob('*.parquet')))}")

Total dataset size: 1.113 GiB
Number of yearly files: 16


In [16]:
from pathlib import Path

data_folder = Path("data")

files = sorted(data_folder.glob("bts_*.parquet"))

print("Files found:", len(files))
print("\nYears found:")

for file in files:
    print(file.name)

print("\nYears in Spark DataFrame:")
flights.select("year").distinct().orderBy("year").show(20)

Files found: 16

Years found:
bts_2010.parquet
bts_2011.parquet
bts_2012.parquet
bts_2013.parquet
bts_2014.parquet
bts_2015.parquet
bts_2016.parquet
bts_2017.parquet
bts_2018.parquet
bts_2019.parquet
bts_2020.parquet
bts_2021.parquet
bts_2022.parquet
bts_2023.parquet
bts_2024.parquet
bts_2025.parquet

Years in Spark DataFrame:
+----+
|year|
+----+
|2025|
+----+



In [17]:
from pyspark.sql.functions import col

flights_clean = flights.filter(
    (col("cancelled") == 0) &
    (col("diverted") == 0) &
    (col("dep_delay").isNotNull()) &
    (col("arr_delay").isNotNull())
)

print("Original flights:", f"{flights.count():,}")
print("Clean flights:", f"{flights_clean.count():,}")

Original flights: 7,001,619
Clean flights: 6,879,484


In [18]:
from pyspark.sql.functions import when, col

flights_clean = flights_clean.withColumn(
    "delayed",
    when(col("arr_delay") >= 15, 1).otherwise(0)
)

flights_clean.select(
    "year",
    "month",
    "op_carrier",
    "origin",
    "dest",
    "arr_delay",
    "delayed"
).show(10)

+----+-----+----------+------+----+---------+-------+
|year|month|op_carrier|origin|dest|arr_delay|delayed|
+----+-----+----------+------+----+---------+-------+
|2025|    1|        AA|   JFK| LAX|     -7.0|      0|
|2025|    1|        AA|   JFK| LAX|      2.0|      0|
|2025|    1|        AA|   JFK| LAX|    -17.0|      0|
|2025|    1|        AA|   JFK| LAX|    -10.0|      0|
|2025|    1|        AA|   JFK| LAX|     -7.0|      0|
|2025|    1|        AA|   JFK| LAX|     -8.0|      0|
|2025|    1|        AA|   JFK| LAX|    -40.0|      0|
|2025|    1|        AA|   JFK| LAX|    -52.0|      0|
|2025|    1|        AA|   JFK| LAX|    -46.0|      0|
|2025|    1|        AA|   JFK| LAX|    -35.0|      0|
+----+-----+----------+------+----+---------+-------+
only showing top 10 rows


In [19]:
from pyspark.sql.functions import count, sum as spark_sum, avg, round

airline_delays = (
    flights_clean
    .groupBy("op_carrier")
    .agg(
        count("*").alias("total_flights"),
        spark_sum("delayed").alias("delayed_flights"),
        round(avg("delayed") * 100, 2).alias("delay_percentage"),
        round(avg("arr_delay"), 2).alias("avg_arrival_delay")
    )
    .orderBy(col("delay_percentage").desc())
)

airline_delays.show(20)

+----------+-------------+---------------+----------------+-----------------+
|op_carrier|total_flights|delayed_flights|delay_percentage|avg_arrival_delay|
+----------+-------------+---------------+----------------+-----------------+
|        F9|       194192|          54202|           27.91|            14.84|
|        OH|       236682|          65158|           27.53|            17.32|
|        B6|       226703|          59288|           26.15|            10.95|
|        AA|       952841|         244989|           25.71|            13.69|
|        G4|       129892|          32428|           24.97|            13.01|
|        AS|       241927|          55887|            23.1|              4.8|
|        YX|       334043|          71857|           21.51|             6.39|
|        UA|       786377|         168815|           21.47|             7.31|
|        NK|       191191|          40931|           21.41|              6.6|
|        WN|      1376837|         294575|            21.4|     

In [20]:
yearly_delays = (
    flights_clean
    .groupBy("year")
    .agg(
        count("*").alias("total_flights"),
        spark_sum("delayed").alias("delayed_flights"),
        round(avg("delayed") * 100, 2).alias("delay_percentage"),
        round(avg("arr_delay"), 2).alias("avg_arrival_delay")
    )
    .orderBy("year")
)

yearly_delays.show(20)

+----+-------------+---------------+----------------+-----------------+
|year|total_flights|delayed_flights|delay_percentage|avg_arrival_delay|
+----+-------------+---------------+----------------+-----------------+
|2025|      6879484|        1534638|           22.31|              8.5|
+----+-------------+---------------+----------------+-----------------+



In [21]:
monthly_delays = (
    flights_clean
    .groupBy("month")
    .agg(
        count("*").alias("total_flights"),
        spark_sum("delayed").alias("delayed_flights"),
        round(avg("delayed") * 100, 2).alias("delay_percentage"),
        round(avg("arr_delay"), 2).alias("avg_arrival_delay")
    )
    .orderBy("month")
)

monthly_delays.show(12)

+-----+-------------+---------------+----------------+-----------------+
|month|total_flights|delayed_flights|delay_percentage|avg_arrival_delay|
+-----+-------------+---------------+----------------+-----------------+
|    1|       522269|          98130|           18.79|             3.76|
|    2|       496476|         103102|           20.77|             5.72|
|    3|       592301|         116034|           19.59|             5.45|
|    4|       577730|         113604|           19.66|             5.66|
|    5|       597574|         141038|            23.6|            10.19|
|    6|       599472|         169405|           28.26|            15.59|
|    7|       612811|         177012|           28.89|            17.35|
|    8|       593733|         133920|           22.56|             9.24|
|    9|       558328|          92859|           16.63|             2.32|
|   10|       601570|         122251|           20.32|             5.59|
|   11|       555296|         114170|           20.

In [22]:
from pyspark.sql import Row

airport_data = [
    Row(airport_code="JFK", airport_name="John F. Kennedy International Airport", city="New York"),
    Row(airport_code="LAX", airport_name="Los Angeles International Airport", city="Los Angeles"),
    Row(airport_code="ORD", airport_name="O'Hare International Airport", city="Chicago"),
    Row(airport_code="ATL", airport_name="Hartsfield-Jackson Atlanta International Airport", city="Atlanta"),
    Row(airport_code="DFW", airport_name="Dallas Fort Worth International Airport", city="Dallas"),
    Row(airport_code="SFO", airport_name="San Francisco International Airport", city="San Francisco"),
    Row(airport_code="SEA", airport_name="Seattle-Tacoma International Airport", city="Seattle"),
    Row(airport_code="MIA", airport_name="Miami International Airport", city="Miami"),
    Row(airport_code="BOS", airport_name="Boston Logan International Airport", city="Boston"),
    Row(airport_code="DEN", airport_name="Denver International Airport", city="Denver")
]

airports = spark.createDataFrame(airport_data)

airports.show()

+------------+--------------------+-------------+
|airport_code|        airport_name|         city|
+------------+--------------------+-------------+
|         JFK|John F. Kennedy I...|     New York|
|         LAX|Los Angeles Inter...|  Los Angeles|
|         ORD|O'Hare Internatio...|      Chicago|
|         ATL|Hartsfield-Jackso...|      Atlanta|
|         DFW|Dallas Fort Worth...|       Dallas|
|         SFO|San Francisco Int...|San Francisco|
|         SEA|Seattle-Tacoma In...|      Seattle|
|         MIA|Miami Internation...|        Miami|
|         BOS|Boston Logan Inte...|       Boston|
|         DEN|Denver Internatio...|       Denver|
+------------+--------------------+-------------+



In [23]:
flights_with_airports = (
    flights_clean
    .join(
        airports,
        flights_clean.origin == airports.airport_code,
        "inner"
    )
    .drop("airport_code")
)

flights_with_airports.select(
    "year",
    "op_carrier",
    "origin",
    "airport_name",
    "city",
    "dest",
    "arr_delay"
).show(10, truncate=False)

+----+----------+------+----------------------------------+------+----+---------+
|year|op_carrier|origin|airport_name                      |city  |dest|arr_delay|
+----+----------+------+----------------------------------+------+----+---------+
|2025|AA        |BOS   |Boston Logan International Airport|Boston|PHL |2.0      |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|PHL |-15.0    |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|PHL |-17.0    |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|PHL |15.0     |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|PHL |-21.0    |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|MIA |-14.0    |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|MIA |7.0      |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|MIA |-7.0     |
|2025|AA        |BOS   |Boston Logan International Airport|Boston|MIA |16.0     |
|2025|AA        

In [24]:
airport_delays = (
    flights_with_airports
    .groupBy("airport_name", "city")
    .agg(
        count("*").alias("total_flights"),
        spark_sum("delayed").alias("delayed_flights"),
        round(avg("delayed") * 100, 2).alias("delay_percentage"),
        round(avg("arr_delay"), 2).alias("avg_arrival_delay")
    )
    .orderBy(col("delay_percentage").desc())
)

airport_delays.show(10, truncate=False)

[Stage 42:======================>                                  (4 + 6) / 10]

+------------------------------------------------+-------------+-------------+---------------+----------------+-----------------+
|airport_name                                    |city         |total_flights|delayed_flights|delay_percentage|avg_arrival_delay|
+------------------------------------------------+-------------+-------------+---------------+----------------+-----------------+
|Dallas Fort Worth International Airport         |Dallas       |306185       |87789          |28.67           |14.94            |
|O'Hare International Airport                    |Chicago      |320418       |84927          |26.51           |12.84            |
|Miami International Airport                     |Miami        |110113       |28671          |26.04           |11.83            |
|Denver International Airport                    |Denver       |314281       |80758          |25.7            |11.15            |
|John F. Kennedy International Airport           |New York     |102370       |25449       

In [25]:
flights_clean.createOrReplaceTempView("flights")

In [26]:
query1 = spark.sql("""
    SELECT *
    FROM flights
    LIMIT 10
""")

query1.show()

+----+-----+------------+-----------+----------+--------+----------+------+----+------------+------------+--------+--------+---------+---------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----------+--------------+-------+
|year|month|day_of_month|day_of_week|op_carrier|tail_num|flight_num|origin|dest|crs_dep_time|crs_arr_time|dep_time|arr_time|dep_delay|arr_delay|cancelled|diverted|carrier_delay|weather_delay|nas_delay|security_delay|late_aircraft_delay|crs_elapsed|actual_elapsed|delayed|
+----+-----+------------+-----------+----------+--------+----------+------+----+------------+------------+--------+--------+---------+---------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----------+--------------+-------+
|2025|    1|           1|          3|        AA|  N104NN|         1|   JFK| LAX|        0659|        1020|    0656|    1013|     -3.0|     -7.0|        0|       0|          0.0|       

In [27]:
query2 = spark.sql("""
    SELECT
        op_carrier,
        COUNT(*) AS total_flights,
        SUM(delayed) AS delayed_flights,
        ROUND(AVG(delayed) * 100, 2) AS delay_percentage
    FROM flights
    GROUP BY op_carrier
    ORDER BY delay_percentage DESC
""")

query2.show(20)

+----------+-------------+---------------+----------------+
|op_carrier|total_flights|delayed_flights|delay_percentage|
+----------+-------------+---------------+----------------+
|        F9|       194192|          54202|           27.91|
|        OH|       236682|          65158|           27.53|
|        B6|       226703|          59288|           26.15|
|        AA|       952841|         244989|           25.71|
|        G4|       129892|          32428|           24.97|
|        AS|       241927|          55887|            23.1|
|        YX|       334043|          71857|           21.51|
|        UA|       786377|         168815|           21.47|
|        NK|       191191|          40931|           21.41|
|        WN|      1376837|         294575|            21.4|
|        MQ|       291835|          61400|           21.04|
|        OO|       825202|         170996|           20.72|
|        DL|      1012427|         200402|           19.79|
|        HA|        79335|          1371

In [28]:
from pyspark.sql.functions import col, when

flights_clean = (
    flights
    .filter(col("cancelled") == 0)
    .filter(col("diverted") == 0)
    .filter(col("dep_delay").isNotNull())
    .filter(col("arr_delay").isNotNull())
    .withColumn(
        "delayed",
        when(col("arr_delay") >= 15, 1).otherwise(0)
    )
)

print("Clean flight records:", f"{flights_clean.count():,}")

Clean flight records: 6,879,484


In [29]:
from pyspark.sql.functions import col

flights_2025 = flights_clean.filter(col("year") == 2025)

selected_flights = flights_2025.select(
    "op_carrier",
    "origin",
    "dest",
    "arr_delay"
)

print("Transformations created.")

Transformations created.


In [30]:
flight_count = selected_flights.count()

print("Number of 2025 flights:", f"{flight_count:,}")

Number of 2025 flights: 6,879,484


In [31]:
selected_flights.explain("formatted")

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [year#0, op_carrier#4, origin#7, dest#8, dep_delay#13, arr_delay#14, cancelled#15, diverted#16]
Batched: true
Location: InMemoryFileIndex [file:/Users/masomashoayb/Desktop/pyspark-nyc-taxi-pipeline/data/bts_2025.parquet]
PushedFilters: [IsNotNull(cancelled), IsNotNull(diverted), IsNotNull(year), EqualTo(cancelled,0), EqualTo(diverted,0), IsNotNull(dep_delay), IsNotNull(arr_delay), EqualTo(year,2025)]
ReadSchema: struct<year:int,op_carrier:string,origin:string,dest:string,dep_delay:double,arr_delay:double,cancelled:int,diverted:int>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [year#0, op_carrier#4, origin#7, dest#8, dep_delay#13, arr_delay#14, cancelled#15, diverted#16]

(3) Filter [codegen id : 1]
Input [8]: [year#0, op_carrier#4, origin#7, dest#8, dep_delay#13, arr_delay#14, cancelled#15, diverted#16]
Condition : (((((((isnotnull(cancelled#15) AND i

In [32]:
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark UI: http://masomas-mbp.lan:4041


In [33]:
import time

start = time.perf_counter()

selected_flights.count()

uncached_time = time.perf_counter() - start

print(f"Uncached execution time: {uncached_time:.2f} seconds")

Uncached execution time: 0.12 seconds


In [34]:
# Cache the selected flight data
cached_flights = selected_flights.cache()

# First action: materialize the cache
start = time.perf_counter()
cached_flights.count()
cache_first_time = time.perf_counter() - start

# Second action: reuse the cached data
start = time.perf_counter()
cached_flights.count()
cache_repeated_time = time.perf_counter() - start

print(f"Uncached time: {uncached_time:.2f} seconds")
print(f"First cached run: {cache_first_time:.2f} seconds")
print(f"Repeated cached run: {cache_repeated_time:.2f} seconds")

Uncached time: 0.12 seconds
First cached run: 1.58 seconds
Repeated cached run: 0.06 seconds


In [35]:
cached_flights.unpersist()

DataFrame[op_carrier: string, origin: string, dest: string, arr_delay: double]

In [38]:
yearly_delays.write.mode("overwrite").parquet("output/yearly_delays")

airline_delays.write.mode("overwrite").parquet("output/airline_delays")

airport_delays.write.mode("overwrite").parquet("output/airport_delays")

print("All analysis results saved successfully!")

All analysis results saved successfully!


In [ ]:
import os

for folder in ["yearly_delays", "airline_delays", "airport_delays"]:
    path = f"output/{folder}"
    print(folder, ":", os.path.exists(path))